# LiDAR processing step 2: look into voxelization & conversion to BEV
- NOTE: Our BEV grid will be (x= [0, 250]m; y: [-100, 100]m)
- we will be using resolution 0.5m 
--> grid resolution is therefore 500 x 400. 
- z∈[-5,3] meters (very typical)

In [3]:
"""
GETTING SET UP FOR BEV_FUSION
conda activate bevfusion38

# Put the path hook INSIDE the env site-packages (not ~/.local)
echo "/home/edgelab/bevfusion" > "$CONDA_PREFIX/lib/python3.8/site-packages/bevfusion_local.pth"

# Verify from env python
python - <<'PY'
import sys, mmdet3d
print(sys.executable)
print(mmdet3d.__file__)
PY

--> restart kernel bevfusion38 --> import sys, site
print(sys.executable)
print("ENABLE_USER_SITE:", site.ENABLE_USER_SITE)

--> import torch, mmcv, mmdet, mmdet3d
print(torch.__version__, mmcv.__version__, mmdet.__version__)
print(mmdet3d.__file__)
"""

'\nGETTING SET UP FOR BEV_FUSION\nconda activate bevfusion38\n\n# Put the path hook INSIDE the env site-packages (not ~/.local)\necho "/home/edgelab/bevfusion" > "$CONDA_PREFIX/lib/python3.8/site-packages/bevfusion_local.pth"\n\n# Verify from env python\npython - <<\'PY\'\nimport sys, mmdet3d\nprint(sys.executable)\nprint(mmdet3d.__file__)\nPY\n\n--> restart kernel bevfusion38 --> import sys, site\nprint(sys.executable)\nprint("ENABLE_USER_SITE:", site.ENABLE_USER_SITE)\n\n--> import torch, mmcv, mmdet, mmdet3d\nprint(torch.__version__, mmcv.__version__, mmdet.__version__)\nprint(mmdet3d.__file__)\n'

In [6]:
import sys, site
print(sys.executable)
print("ENABLE_USER_SITE:", site.ENABLE_USER_SITE)

import torch, mmcv, mmdet, mmdet3d
print(torch.__version__, mmcv.__version__, mmdet.__version__)
print(mmdet3d.__file__)

/home/edgelab/miniconda3/envs/bevfusion38/bin/python
ENABLE_USER_SITE: True
1.10.2 1.4.0 2.20.0
/home/edgelab/bevfusion/mmdet3d/__init__.py


In [7]:
#GET point cloud data from BEV_Fusion.ipynbimport numpy as np
# variable fov_points_cam_xyzi is defined in BEV_Fusion.ipynb
# it is a numpy array of shape (N, 4)
# the 4 columns are: x, y, z, intensity
# it is both denoised, restricted to camera FOV, and points are in ego frame
from pathlib import Path
import numpy as np

p = Path("/home/edgelab/multimodal-MoE/outputs/tmp/fov_points_cam_xyzi.npy")
fov_points_cam_xyzi = np.load(p)
print("loaded:", p, fov_points_cam_xyzi.shape, fov_points_cam_xyzi.dtype)
#fov_points_cam_xyzi is now loaded into memory
# we can now use it for LiDAR processing as in BEV_FUSION paper
# ------------------------------------------------------------
# LiDAR processing step 2: voxelization & conversion to BEV
# ------------------------------------------------------------
# voxelization:
# - convert point cloud to voxel grid
# - each voxel contains a set of points
# - this is useful for 3D object detection
# - we will use a voxel size of 0.5m
# - we will use a voxel grid size of 250m x 200m


loaded: /home/edgelab/multimodal-MoE/outputs/tmp/fov_points_cam_xyzi.npy (81286, 4) float32


# Now we have 